<a href="https://colab.research.google.com/github/jasonwong-lab/BIOF3001/blob/main/BIOF3001_SigProfiler_Demonstration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BIOF3001: Mutational Processes Workshop (SigProfiler)

22nd September 2026

Michelle Huang & Jason Wong

---

# Computing Mutational Signatures from Cancer Genomes
In this workshop, we will explore the use of the SigProfiler package [SigProfilerMatrixGenerator](https://github.com/SigProfilerSuite/SigProfilerMatrixGenerator) and [SigProfilerAssignment](https://github.com/SigProfilerSuite/SigProfilerAssignment/tree/main) to analyse somatic mutations in cancer genomes.


We will first use curated cancer somatic-mutation VCF files
to generate SBS96 mutational profiles, then use known COSMIC mutational signatures to estimate which mutational processes contribute to each tumour sample. Finally, we will
visualise and compare SBS96 mutation profiles across cancer types, and identify samples with unusual mutation spectra.


### Dataset
The full research project analyses more than 15,000 tumour genomes from both The Cancer Genome Atlas (TCGA) and non-TCGA cancer-genomics resources to identify tumour samples with unusual mutational-signature profiles and to investigate whether these patterns may reflect biological processes, exposure
history, DNA-repair defects, or technical artefacts. For this workshop, we use a small curated subset of 48 tumour VCFs for illustration. All VCF files were filtered to contain PASS variants and single-base substitutions only.




## 1. Install and import required software

We use:

- `SigProfilerMatrixGenerator` to generate SBS96 mutational matrices
- `SigProfilerAssignment` to assign COSMIC mutational signatures
- Python packages for data handling, PCA, and visualisation


In [ ]:
# Install SigProfiler packages
!pip install --quiet SigProfilerMatrixGenerator
!pip install --quiet SigProfilerAssignment

In [ ]:
# Load required libraries

# Built-in Python modules
import os
import random

# Data handling and visualisation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# SigProfiler packages
from SigProfilerMatrixGenerator import install as genInstall
from SigProfilerMatrixGenerator.scripts import SigProfilerMatrixGeneratorFunc as matGen
from SigProfilerAssignment import Analyzer as Analyze

# Handing Google Drive downloads
!pip install gdown

# Download files
from google.colab import files

# Set seeds for reproducibility
seed = 42
np.random.seed(seed)
random.seed(seed)

print("All libraries imported successfully.")

To classify VCF mutations into SBS96 categories, SigProfiler also requires the human reference genome sequence (~2 minutes).

In [ ]:
# Install the GRCh38 human reference genome
genInstall.install("GRCh38")

print("GRCh38 human reference genome installed successfully.")

# 2. Load VCF files from Google Drive



###Mount Google Drive

In [ ]:
# Google import
from google.colab import drive
drive.mount('/content/drive/')

### Create working directory and download VCF files

In [ ]:
import os
try:
  os.mkdir("/content/drive/MyDrive/BIOF3001_SigProfiler")
except FileExistsError:
  print("directory already exist. OK to continue")
os.chdir("/content/drive/MyDrive/BIOF3001_SigProfiler")

# Download VCF file gzip
!gdown "https://drive.google.com/uc?export=download&id=1gHDWjF3xDubew6RU7KkXh_5WCS2JM3IY" --output /content/drive/MyDrive/BIOF3001_SigProfiler/sigprofiler_example_vcf.tar.gz

# Unzip VCF files
!tar -zxvf /content/drive/MyDrive/BIOF3001_SigProfiler/sigprofiler_example_vcf.tar.gz

###Define project folders

In [ ]:
# Define the working directory
PROJECT_DIR = "/content/drive/MyDrive/BIOF3001_SigProfiler"

# Input VCF files
VCF_DIR = "/content/drive/MyDrive/BIOF3001_SigProfiler/VCF"

# MatrixGenerator automatically writes output inside the VCF folder
MATRIX_OUTPUT_DIR = "/content/drive/MyDrive/BIOF3001_SigProfiler/VCF/output"

# SigProfilerAssignment output folder
ASSIGNMENT_OUTPUT_DIR = (
    "/content/drive/MyDrive/BIOF3001_SigProfiler/SigProfilerAssignment"
)

# Create SigProfilerAssignment output folder
os.makedirs(ASSIGNMENT_OUTPUT_DIR, exist_ok=True)

# 3. SigProfilerMatrixGenerator: Generate SBS96 mutational matrix


The package that we will use is [SigProfilerMatrixGenerator](https://github.com/SigProfilerSuite/SigProfilerMatrixGenerator), it uses the VCF files and the GRCh38 reference genome to generate an SBS96 mutation matrix. Rows represent mutation channels (i.e. each of the 96 mutational trinucleotide context, A[C>A]A, A[C>A]C, etc). Columns
represent tumour samples, and each value is the number of mutations observed in that channel.

Processing the 48 VCF files will take around **6 minutes**.

In [ ]:
# Run SigProfilerMatrixGenerator
matrices = matGen.SigProfilerMatrixGeneratorFunc(
    project="MatrixGenerator",
    reference_genome="GRCh38",
    path_to_input_files=VCF_DIR,
    plot=True,
)

### Inspection of the output

In this workshop, we focus on the SBS96 matrix (i.e. trinucleotide spectrum). It contains 96 possible trinucleotide mutation frequencies across all tumour samples.

We first inspect the dimensions and first few rows of the SBS96 matrix.

In [ ]:
# Load the SBS96 matrix generated by SigProfilerMatrixGenerator
SBS96_FILE = os.path.join(
    MATRIX_OUTPUT_DIR,
    "SBS",
    "MatrixGenerator.SBS96.all"
)

sbs96 = pd.read_csv(
    SBS96_FILE,
    sep="\t",
    index_col=0
)

print(sbs96.shape)
sbs96.iloc[:10, :6]

Download the SBS96 mutational spectrum plots generated by SigProfilerMatrixGenerator. Compare the mutation profiles across the tumour samples.

This spectrum is the mutation profile that SigProfiler will use for signature assignment.

In [ ]:
# Download the PDF containing SBS96 mutational spectra for all samples
SBS96_PLOT = os.path.join(
    MATRIX_OUTPUT_DIR,
    "plots",
    "SBS_96_plots_MatrixGenerator.pdf"
)

files.download(SBS96_PLOT)

# 4. SigProfilerAssignment: Assign COSMIC mutational signatures

We now use [SigProfilerAssignment](https://github.com/SigProfilerSuite/SigProfilerAssignment/tree/main) to assign known COSMIC mutational signatures to each tumour sample. SigProfilerAssignment uses the SBS96 mutation matrix generated previously.

For each tumour sample, SigProfilerAssignment estimates:

- the activity of each COSMIC SBS signature
- the reconstructed SBS96 profile based on these activities
- reconstruction metrics that describe how well the assigned signatures explain
  the observed mutation profile

In this workshop, we use [COSMIC mutational signatures](https://cancer.sanger.ac.uk/signatures/sbs/) version 3.6.

This step takes around **2 minutes** to complete.

In [ ]:
# Define the path to the SBS96 matrix generated by SigProfilerMatrixGenerator
SBS96_FILE = os.path.join(
    MATRIX_OUTPUT_DIR,
    "SBS",
    "MatrixGenerator.SBS96.all"
)

# Run SigProfilerAssignment
Analyze.cosmic_fit(
    samples=SBS96_FILE,
    output=ASSIGNMENT_OUTPUT_DIR,
    input_type="matrix",
    context_type="96",
    genome_build="GRCh38",
    cosmic_version=3.6,
    make_plots=True,
    verbose=True
)

### Inspection of the output

SigProfilerAssignment generates several output files, including estimated
signature activities for each sample, the reference signatures used for fitting,
and statistics describing how well the fitted signature combination reconstructs
each sample's observed SBS96 spectrum.

We first inspect the the two main result tables.

The activity table gives the estimated number of mutations attributed to each COSMIC signature in each tumour sample. The sample statistics table reports how well the fitted signatures reconstruct the observed SBS96 mutation profile.

In [ ]:
# Define the path to SigProfilerAssignment results folder
ASSIGNMENT_SOLUTION_DIR = os.path.join(
    ASSIGNMENT_OUTPUT_DIR,
    "Assignment_Solution"
)

# Load estimated signature activities
ACTIVITIES_FILE = os.path.join(
    ASSIGNMENT_SOLUTION_DIR,
    "Activities",
    "Assignment_Solution_Activities.txt"
)

# Load sample-fit statistics
STATS_FILE = os.path.join(
    ASSIGNMENT_SOLUTION_DIR,
    "Solution_Stats",
    "Assignment_Solution_Samples_Stats.txt"
)

activities = pd.read_csv(ACTIVITIES_FILE, sep="\t")
sample_stats = pd.read_csv(STATS_FILE, sep="\t")

print("Activities matrix:", activities.shape)
display(activities.head())

print("\nSample-fit statistics:", sample_stats.shape)
display(sample_stats.head())



We can visualise the estimated contribution of each mutational signature across
all samples using the activity plot generated by SigProfilerAssignment.

Download and inspect the activity plot.

In [ ]:
# Download the COSMIC SBS signature activity plot
ACTIVITY_PLOT = os.path.join(
    ASSIGNMENT_SOLUTION_DIR,
    "Activities",
    "Assignment_Solution_Activity_Plots.pdf"
)

files.download(ACTIVITY_PLOT)

# 5. Outlier screening
Tumour samples can differ both in total mutation burden and mutation-profile pattern. To compare signatures we should first convert each SBS96 profile into mutation proportions (i.e. sum of all 96 trinucleotide fractions = 1)

In [ ]:
# Load SBS96 mutation matrix
sbs96_matrix = pd.read_csv(
    SBS96_FILE,
    sep="\t",
    index_col=0
)

# Transpose matrix so that rows are samples and columns are SBS96 mutation types
sbs96_samples = sbs96_matrix.T

# Convert mutation counts to proportions within each sample
sbs96_profiles = sbs96_samples.div(
    sbs96_samples.sum(axis=1),
    axis=0
)


### PCA Plotting

PCA reduces the 96 SBS trinucleotide spectra into
two dimensions for visualisation. Samples with similar SBS96 profiles tend to appear closer together.

In [ ]:
%matplotlib inline

# Define tumour groups from sample names
group = []

for sample_name in sbs96_profiles.index:

    if sample_name.startswith("TCGA_BRCA"):
        group.append("BRCA")
    elif sample_name.startswith("TCGA_COAD"):
        group.append("COAD")
    elif sample_name.startswith("TCGA_ESCA"):
        group.append("ESCA")
    elif sample_name.startswith("TCGA_SKCM"):
        group.append("SKCM")
    elif sample_name.startswith("TCGA_STAD"):
        group.append("STAD")
    else:
        group.append("Sample_X")

# Run PCA using all 96 SBS mutation channels
pca = PCA(n_components=2)
pca_results = pca.fit_transform(sbs96_profiles)

# Calculate percentage of variation explained
var_expl = (
    100 * pca.explained_variance_ratio_[:2]
).round(1)

# Create a data frame for plotting
df_pca = pd.DataFrame(
    {
        "sample": sbs96_profiles.index,
        "group": group,
        "PC1": pca_results[:, 0],
        "PC2": pca_results[:, 1]
    }
)

# Plot PCA results
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=df_pca,
    x="PC1",
    y="PC2",
    hue="group",
    s=60
)


plt.xlabel(f"PC1 ({var_expl[0]}%)")
plt.ylabel(f"PC2 ({var_expl[1]}%)")
plt.title("PCA of SBS96 mutational profiles")

plt.tight_layout()
plt.show()

### Identifying outlier samples

Examine the PCA plot. Samples at the extremes of PC1 or PC2 may have unusual SBS96 mutation profiles.

The following code uses the Local Outlier Factor (LOF) method to detect outliers that are distinct from the nearest 5 samples based on cosine distance.



In [ ]:
import pandas as pd
from sklearn.neighbors import LocalOutlierFactor

# LOF looks at local density using Cosine distance for angular/proportional data
# Note: LOF uses 'novelty=False' by default for outlier detection on training data
lof = LocalOutlierFactor(n_neighbors=5, metric='cosine', contamination=0.05)
predictions = lof.fit_predict(sbs96_profiles)

# Extract outliers
outlier_samples = sbs96_profiles.index[predictions == -1].tolist()
print("Outlier samples (Cosine LOF):", outlier_samples)


Do the outliers that the LOF method suggest make sense?

In [ ]:
# Plot PCA again this time labelling outliers found by LOF
group = []

for sample_name in sbs96_profiles.index:

    if sample_name.startswith("TCGA_BRCA"):
        group.append("BRCA")
    elif sample_name.startswith("TCGA_COAD"):
        group.append("COAD")
    elif sample_name.startswith("TCGA_ESCA"):
        group.append("ESCA")
    elif sample_name.startswith("TCGA_SKCM"):
        group.append("SKCM")
    elif sample_name.startswith("TCGA_STAD"):
        group.append("STAD")
    else:
        group.append("Sample_X")

# Run PCA using all 96 SBS mutation channels
pca = PCA(n_components=2)
pca_results = pca.fit_transform(sbs96_profiles)

# Calculate percentage of variation explained
var_expl = (
    100 * pca.explained_variance_ratio_[:2]
).round(1)

# Create a data frame for plotting
df_pca = pd.DataFrame(
    {
        "sample": sbs96_profiles.index,
        "group": group,
        "PC1": pca_results[:, 0],
        "PC2": pca_results[:, 1]
    }
)

# Plot PCA results
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=df_pca,
    x="PC1",
    y="PC2",
    hue="group",
    s=60
)

# Loop through the rows that match your target samples
for idx, row in df_pca[df_pca['sample'].isin(outlier_samples)].iterrows():
    plt.annotate(
        row['sample'],                    # The text to display
        xy=(row['PC1'], row['PC2']),      # Target point coordinates
        xytext=(12, 4),                    # Offset text by 5 points right and 5 points up
        textcoords='offset points',       # Interpret xytext as pixel/point offsets
        fontsize=9,                       # Text size
        weight='bold',                    # Make it stand out
        color='black',                     # Label color
        arrowprops=dict(
            arrowstyle="-",               # A clean, simple line (no arrow head)
            color="black",                # Color of the line
            linestyle="-",               # Dashed line (use "-" for solid)
            linewidth=0.8,                # Thickness of the line
            # 3. 'shrinkA=0' forces the line to touch the text box perfectly
            shrinkA=0,
            # 4. 'shrinkB=2' stops the line 2 pixels short of the dot so it doesn't overlap the marker
            shrinkB=5,
            connectionstyle="arc3,rad=0"
        )
    )

plt.xlabel(f"PC1 ({var_expl[0]}%)")
plt.ylabel(f"PC2 ({var_expl[1]}%)")
plt.title("PCA of SBS96 mutational profiles")

plt.tight_layout()
plt.show()

### Inspecting signature activities

Select one or two samples that appear distinct in the PCA plot. We will inspect their non-zero COSMIC signature activities to determine which mutational signatures may contribute to their unusual SBS96 profiles.

Replace the example sample names below with samples you identified from the PCA plot.


In [ ]:
# Replace these example names with samples selected from the PCA plot

# View non-zero signature activities for each selected sample
for sample_name in outlier_samples:

    print(f"\n{sample_name}")

    sample_activities = (
        activities[activities["Samples"] == sample_name]
        .set_index("Samples")
        .T
    )

    sample_activities.columns = ["Attributed mutations"]

    sample_activities = (
        sample_activities[
            sample_activities["Attributed mutations"] > 0
        ]
        .sort_values("Attributed mutations", ascending=False)
    )

    display(sample_activities)

We can also inspect the SBS96 mutational spectrum of the selected samples to identify the mutation pattern responsible for their unusual PCA positions.

Download the SBS96 mutational-spectrum PDF generated by SigProfilerMatrixGenerator if you have not.

In [ ]:
# Download the PDF containing SBS96 mutational spectra for all samples
SBS96_PLOT = os.path.join(
    MATRIX_OUTPUT_DIR,
    "plots",
    "SBS_96_plots_MatrixGenerator.pdf"
)

files.download(SBS96_PLOT)